Hyperparams selection

# Libraries

In [82]:
import openpyxl
import pandas as pd
from pathlib import Path
import numpy as np
import optuna
import catboost
from sklearn.metrics import f1_score
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from imblearn.over_sampling import SMOTE

# Datatasets selection

In [60]:
path_data = Path('../data')
path_results = Path('../results/')
path_highlighted = path_results / 'metrics_modelling2_highlight.xlsx'
df_highlighted = pd.read_excel(path_highlighted)
wb = openpyxl.load_workbook(path_highlighted, data_only=True)
ws = wb.worksheets[0]

In [66]:
highlighted_rows = []
for row in range(1, ws.max_row):
    cell = ws.cell(column=1, row=row)
    bgColor = cell.fill.bgColor.index
    fgColor = cell.fill.fgColor.index
    if bgColor != '00000000':
        highlighted_rows.append(row)
highlighted_rows = np.array(highlighted_rows) - 2
df_filtered = df_highlighted.iloc[highlighted_rows]

Datasets for parameters selection.

In [74]:
df_filtered

,dataset,target,model,accuracy,f1,precision,recall,roc_auc
2,df_modelling_no_multicollinearity,splashing,catboostclassifier_splashing,0.866667,0.900000,0.865385,0.937500,0.839120
3,df_modelling_dimensionless,splashing,kneighborsclassifier_splashing_ordenc,0.866667,0.900000,0.865385,0.937500,0.839120
4,df_modelling_dimensionless,splashing,catboostclassifier_splashing,0.866667,0.900000,0.865385,0.937500,0.839120
5,df_modelling_dimensionless,net_impact,catboostclassifier_net_impact,0.946667,0.900000,0.900000,0.900000,0.931818
6,df_modelling_dimensionless,splashing,kneighborsclassifier_smote_splashing_ordenc,0.866667,0.897959,0.880000,0.916667,0.847222
7,df_modelling_dimensionless,splashing,kneighborsclassifier_splashing_onehot,0.853333,0.891089,0.849057,0.937500,0.820602
8,df_modelling_dimensionless,splashing,svc_smote_splashing_onehot,0.853333,0.888889,0.862745,0.916667,0.828704
11,df_modelling_no_multicollinearity,splashing,catboostclassifier_smote_splashing,0.853333,0.886598,0.877551,0.895833,0.836806
12,df_modelling_dimensionless,splashing,catboostclassifier_smote_splashing,0.853333,0.886598,0.877551,0.895833,0.836806
13,df_modelling_dimensionless,splashing,logisticregression_splashing_onehot,0.840000,0.880000,0.846154,0.916667,0.810185


In [79]:
sorted(df_filtered['model'].unique())

['catboostclassifier_net_impact',
 'catboostclassifier_smote_net_impact',
 'catboostclassifier_smote_splashing',
 'catboostclassifier_splashing',
 'kneighborsclassifier_net_impact_onehot',
 'kneighborsclassifier_net_impact_ordenc',
 'kneighborsclassifier_smote_net_impact_ordenc',
 'kneighborsclassifier_smote_splashing_ordenc',
 'kneighborsclassifier_splashing_onehot',
 'kneighborsclassifier_splashing_ordenc',
 'logisticregression_net_impact_ordenc',
 'logisticregression_splashing_onehot',
 'svc_smote_net_impact_onehot',
 'svc_smote_net_impact_ordenc',
 'svc_smote_splashing_onehot',
 'svc_smote_splashing_ordenc']

# Parameters selection

In [ ]:
def get_params(trial, model_str):
    if 'catboostclassifier' in model_str:
        params = {
            "objective": trial.suggest_categorical(
                "objective", ["Logloss", "CrossEntropy"]),
            "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.01, 0.1),
            "depth": trial.suggest_int("depth", 1, 12),
            "boosting_type": trial.suggest_categorical(
                "boosting_type", ["Ordered", "Plain"]),
            "bootstrap_type": trial.suggest_categorical(
                "bootstrap_type", ["Bayesian", "Bernoulli", "MVS"])}
        if params["bootstrap_type"] == "Bayesian":
            params["bagging_temperature"] = trial.suggest_float(
                "bagging_temperature", 0, 10)
        elif params["bootstrap_type"] == "Bernoulli":
            params["subsample"] = trial.suggest_float("subsample", 0.1, 1)
    if 'kneighborsclassifier' in model_str:
        params = {
            "algorithm": trial.suggest_categorical(
                "algorithm", ["auto", "ball_tree", "kd_tree", "brute"]
            ),
            "n_neighbors": trial.suggest_int("n_neighbors", 1, 30),
            "weights": trial.suggest_categorical("weights", ["uniform", "distance"]),
            "metric": trial.suggest_categorical("metric", 
                                        ["euclidean", "manhattan", "minkowski"])
        }
    if 'svc' in model_str:
        params = {
            'C': trial.suggest_loguniform('C', 1e-5, 1e5),
            'kernel': trial.suggest_categorical('kernel', ['linear', 'poly', 'rbf', 'sigmoid']),
            'gamma': trial.suggest_loguniform('gamma', 1e-5, 1e5),
            'degree': trial.suggest_int('degree', 2, 5),
            'coef0': trial.suggest_uniform('coef0', -1.0, 1.0)}
    if 'logisticregression' in model_str:
        params = {
            'penalty': trial.suggest_categorical('penalty', ['l1', 'l2']),
            'C': trial.suggest_loguniform('C', 0.001, 10),
            'solver': trial.suggest_categorical('solver', ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']),
            'max_iter': trial.suggest_int('max_iter', 100, 300),
            'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced']),
            'multi_class': trial.suggest_categorical('multi_class', ['auto', 'ovr', 'multinomial'])
        }
    if 'xgbclassifier' in model_str: 
        params = {
            'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'gamma': trial.suggest_float('gamma', 0, 1),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'reg_alpha': trial.suggest_float('reg_alpha', 0, 10),
            'reg_lambda': trial.suggest_float('reg_lambda', 0, 10),
        }
    return params

def get_pipeline(model_str, num_features, random_state, cat_features=['wettability']):
    transformers = []
    if ('xgbclassifier' not in model_str) and ('catboostclassifier' not in model_str):
        numeric_transformer = Pipeline(steps=[("scaler", StandardScaler())])
        transformers = [("num", numeric_transformer, num_features)]
    if 'onehot' in model_str:
        categorical_transformer = Pipeline(
            steps=[('onehot', OneHotEncoder(handle_unknown="ignore"))])
        transformers.append(('cat', categorical_transformer, cat_features))
    if 'ordenc' in model_str:
        categorical_transformer = Pipeline(
            steps=[('ordenc', OrdinalEncoder(handle_unknown='use_encoded_value', 
                                     unknown_value=np.nan))])
        transformers.append(('cat', categorical_transformer, cat_features))
    if 'smote' in model_str:
        smt = SMOTE(random_state=random_state)

In [ ]:
def objective(
        trial, train, test, target, model_str, cat_features=['wettability']):
    params = create_params(trial, model_str)
    gbm = catboost.CatBoostClassifier(**params, cat_features=cat_features)
    gbm.fit(train.drop(columns=[target]), train[target])
    preds = gbm.predict(test.drop(columns=[target]))
    f1 = f1_score(test[target], preds)
    return f1

In [ ]:
study = optuna.create_study(direction="maximize")
